# ΣΔ Decimator — Analysis Notebook

Stage 2 of the Trouper DSP chain. Converts each SX1257 1-bit complex ΣΔ bitstream
(32 MS/s) into signed 8-bit IQ samples for the on-chip receive path.

**Deployed configuration:** CIC-only (N=3), `decim_ratio=1` → R=128, fs_out=250 kS/s for both 125 kHz and 250 kHz LoRa bandwidths.

**Reference:** `planning/blocks/ΣΔ Decimator.md`, `planning/cic-only-decimator-findings.md`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter

from sim.models.decimator import (
    SigmaDeltaDecimator, decimation_ratio, FS_ADC, FIR_COEFFS, RATIO_FOR_BW
)
from sim.models.lora import modulate

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print(f'FS_ADC = {FS_ADC/1e6:.0f} MS/s')
print(f'Supported BWs → ratios: {RATIO_FOR_BW}')

---
## 1  CIC Transfer Function

The 3-stage CIC decimator has the transfer function:

$$H(z) = \left[\frac{1 - z^{-R}}{1 - z^{-1}}\right]^3$$

The normalised magnitude response (after dividing by $R^3$) is:

$$|H_{\text{norm}}(f)| = \left|\frac{\sin(\pi f R / f_s)}{R \sin(\pi f / f_s)}\right|^3$$

At the deployed R=128, the −3 dB point is near 0.26×Nyquist. Droop at 0.4×Nyquist (50 kHz for 250 kHz LoRa) is about **−1.7 dB** — well within the LoRa signal band and tolerable for the 125/250 kHz BWs. The optional 9-tap FIR compensation buys back ~7 dB at that frequency (RTL: CIC-only 30.6 dB → CIC+FIR 37.7 dB SQNR).

In [ ]:
def cic_response(R: int, N: int, n_freqs: int = 4096) -> tuple:
    """Normalised CIC magnitude response at input sample rate (dB)."""
    # Evaluate at f/fs_in = [0, 1/(2R)] (the output Nyquist band)
    f_norm = np.linspace(1e-9, 0.5 / R, n_freqs)
    # |H_norm(f)| = |sin(π*R*f_norm) / (R * sin(π*f_norm))|^N
    h = np.abs(np.sin(np.pi * R * f_norm) / (R * np.sin(np.pi * f_norm))) ** N
    return f_norm * R * 2, 20 * np.log10(np.maximum(h, 1e-10))  # x-axis in units of Nyquist fraction

def cic_plus_fir_response(R: int, N: int, n_freqs: int = 4096) -> tuple:
    """CIC magnitude + 9-tap FIR compensation, both evaluated at output rate."""
    f_frac, h_cic_db = cic_response(R, N, n_freqs)
    # FIR response at output rate: H_fir(f) = sum(h[n] * exp(-j2π*f*n))
    f_out_norm = f_frac / 2  # convert to fraction of output Nyquist (=0.5)
    n_taps = len(FIR_COEFFS)
    n_arr = np.arange(n_taps)
    H_fir = np.array([np.sum(FIR_COEFFS * np.exp(-2j * np.pi * f * n_arr)) for f in f_out_norm])
    h_total_db = h_cic_db + 20 * np.log10(np.maximum(np.abs(H_fir), 1e-10))
    return f_frac, h_total_db

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, R, title_bw in zip(axes, [128, 64], ['R=128 (deployed: 125/250 kHz)', 'R=64 (500 kHz — not supported)']):
    f_frac, h_cic = cic_response(R, N=3)
    f_frac, h_comp = cic_plus_fir_response(R, N=3)
    ax.plot(f_frac, h_cic,  'b-',  lw=2, label='CIC-only (deployed)')
    ax.plot(f_frac, h_comp, 'r--', lw=2, label='CIC + FIR compensation')
    ax.axvline(0.4, color='gray', ls=':', lw=1.5, label='0.4×Nyquist (SQNR test point)')
    ax.axhline(-3, color='k', ls=':', lw=1, alpha=0.4)
    ax.set_xlim(0, 1)
    ax.set_ylim(-25, 2)
    ax.set_xlabel('Frequency (fraction of output Nyquist = fs_out/2)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(title_bw)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)
    # Annotate droop at 0.4×Nyquist
    idx = np.argmin(np.abs(f_frac - 0.4))
    ax.annotate(f'{h_cic[idx]:.1f} dB', xy=(0.4, h_cic[idx]), xytext=(0.5, h_cic[idx]+3),
                arrowprops=dict(arrowstyle='->', color='blue'), color='blue', fontsize=8)

fig.suptitle('CIC³ Frequency Response — Normalised Passband', fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/sd_decimator_response.png', bbox_inches='tight')
plt.show()
print('Saved: sim/plots/sd_decimator_response.png')

---
## 2  SQNR Measurement

SQNR is measured by feeding a 1st-order ΣΔ-modulated tone at **0.4×Nyquist** through the CIC decimator and fitting a sine to the output.

**RTL benchmark** (from `planning/cic-only-decimator-findings.md`, job 1102):

| R | CIC-only SQNR (RTL, authoritative) | Pass (≥28 dB)? |
|---|---|---|
| 256 | 27.2 dB | ✗ |
| 128 | 30.6 dB | **✓** |
| 64  | 9.6 dB  | ✗ |
| 32  | 1.8 dB  | ✗ |

> **Warning — Python model is not SQNR-accurate:**
> The floating-point Python CIC model does **not** model ΣΔ alias noise. At R=64 the RTL shows 9.6 dB but the Python model gives ~70 dB — a difference caused entirely by quantisation noise aliases that fold through the CIC at low OSR. Even at R=128 the Python model predicts ~70 dB vs the RTL's 30.6 dB because float precision avoids all output quantisation noise.
>
> The Python SQNR values below are included only to confirm the measurement code runs end-to-end. **Always use RTL simulation for SQNR decisions.**

In [ ]:
def sigma_delta_1st_order(x: np.ndarray) -> np.ndarray:
    """1st-order ΣΔ modulator. Input must be |x| < 1 for stability. Returns ±1."""
    out = np.empty_like(x)
    e = 0.0
    for i, xi in enumerate(x):
        u = xi + e
        q = 1.0 if u >= 0.0 else -1.0
        e = u - q
        out[i] = q
    return out


def measure_sqnr(R: int, n_periods: int = 500, amp: float = 0.9) -> float:
    """
    Feed a tone at 0.4×Nyquist through ΣΔ → CIC, fit a sine, return SQNR in dB.
    """
    fs_in = FS_ADC
    fs_out = fs_in / R
    f_tone = 0.4 * fs_out / 2   # 0.4 × Nyquist in Hz

    # Use enough input samples for clean frequency resolution
    n_out = n_periods * int(round(fs_out / f_tone))
    n_in  = n_out * R
    t_in  = np.arange(n_in) / fs_in

    # ΣΔ modulate: real tone only (I channel); Q stays zero
    tone  = amp * np.sin(2 * np.pi * f_tone * t_in)
    bits  = sigma_delta_1st_order(tone) + 0j  # complex with zero Q for this test

    dec   = SigmaDeltaDecimator(ratio=R, output_bits=16, cic_only=True)
    out   = dec.process(bits).real

    # Discard initial transient (CIC group delay ≈ N*(R-1)/2 input samples)
    skip = int(3 * R * 3 / fs_out * fs_out) + 10  # ~3× CIC settling at output rate
    y    = out[skip:]
    n    = np.arange(len(y))

    # LS sine fit: A*cos(ω*n) + B*sin(ω*n)
    omega = 2 * np.pi * f_tone / fs_out
    C = np.column_stack([np.cos(omega * n), np.sin(omega * n)])
    coeffs, _, _, _ = np.linalg.lstsq(C, y, rcond=None)
    fitted  = C @ coeffs
    signal_amp = np.sqrt(coeffs[0]**2 + coeffs[1]**2)
    noise_rms  = float(np.std(y - fitted))

    sqnr = 20 * np.log10(signal_amp / noise_rms) if noise_rms > 0 else float('inf')
    return sqnr, signal_amp, noise_rms


# RTL benchmark from planning/cic-only-decimator-findings.md (job 1102)
rtl_sqnr = {256: 27.2, 128: 30.6, 64: 9.6, 32: 1.8}

print(f'{"R":>5}  {"BW":>8}  {"fs_out":>10}  {"Python SQNR":>12}  {"RTL SQNR":>10}  {"Pass (≥28dB)":>13}')
print('-' * 68)
results = {}
for R in [256, 128, 64, 32]:
    sqnr_py, sig, nse = measure_sqnr(R)
    rtl = rtl_sqnr.get(R, float('nan'))
    bw_hz = FS_ADC / R / 2
    fs_out = FS_ADC / R
    passed = '✓' if rtl >= 28 else '✗'
    results[R] = sqnr_py
    print(f'{R:>5}  {bw_hz/1e3:>7.0f}kHz  {fs_out/1e3:>8.0f}kS/s  '
          f'{sqnr_py:>10.1f} dB  {rtl:>8.1f} dB  {passed:>13}')

print()
print('Note: Python model over-estimates SQNR at R=64 and R=32 because')
print('floating-point CIC does not model ΣΔ alias noise accurately.')
print('RTL measurement is authoritative. See cic-only-decimator-findings.md.')

---
## 3  Why R=64 Fails — Alias Noise

At R=64, ΣΔ quantisation noise aliases back into the signal band because the CIC null at DC (R×fs_tone) falls too close to the passband. The Python model misses this; the RTL shows 9.6 dB. Compare the output spectra.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)

for ax, R, color, note in [
    (axes[0], 128, 'tab:blue', 'R=128 (deployed, SQNR≈30.6 dB RTL) ✓'),
    (axes[1],  64, 'tab:red',  'R=64  (unsupported, SQNR≈9.6 dB RTL) ✗'),
]:
    fs_out  = FS_ADC / R
    f_tone  = 0.4 * fs_out / 2
    n_out   = 2048
    t_in    = np.arange(n_out * R) / FS_ADC
    tone    = 0.9 * np.sin(2 * np.pi * f_tone * t_in)
    bits    = sigma_delta_1st_order(tone) + 0j
    dec     = SigmaDeltaDecimator(ratio=R, output_bits=16, cic_only=True)
    out     = dec.process(bits).real[-n_out:]

    # Power spectrum
    N_fft   = len(out)
    window  = np.hanning(N_fft)
    S       = np.abs(np.fft.rfft(out * window)) ** 2
    S_db    = 10 * np.log10(S / S.max() + 1e-12)
    freqs   = np.fft.rfftfreq(N_fft) * fs_out / 1e3   # kHz

    ax.plot(freqs, S_db, color=color, lw=0.8)
    ax.axvline(f_tone / 1e3, color='k', ls='--', lw=1, label=f'Tone: {f_tone/1e3:.1f} kHz')
    ax.set_xlim(0, fs_out / 2e3)
    ax.set_ylim(-80, 5)
    ax.set_xlabel('Frequency (kHz)')
    ax.set_ylabel('Power (dB, normalised to peak)')
    ax.set_title(note)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

fig.suptitle('CIC Output Spectrum — R=128 vs R=64 (Python model)', fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/sd_decimator_spectra.png', bbox_inches='tight')
plt.show()

---
## 4  Supported Operating Modes

Both supported LoRa bandwidths share a single firmware setting: `decim_ratio=1` → R=128 → 250 kS/s output.

For 125 kHz BW, this is intentionally **2× oversampled**. The SX1257 analog IF filter already band-limits the input; the SX1302 downstream channel filter rejects the extra bandwidth. The sd_remod → SX1302 path provides a third stage of filtering.

In [ ]:
from sim.models.decimator import RATIO_1MS

rows = [
    # (LoRa BW, decim_ratio, R, fs_out kS/s, status)
    (125e3,  1, 128, 250,  '✓ Supported (2× oversampled — SX1302 re-filters)'),
    (250e3,  1, 128, 250,  '✓ Supported (1× Nyquist)'),
    (500e3,  2,  64, 500,  '✗ NOT SUPPORTED — CIC SQNR only 9.6 dB'),
    (500e3,  3,  32, 1000, '⚠ Debug/lab only (1 MS/s, 2× oversampled 500 kHz)'),
    (125e3,  0, 256, 125,  '⚠ Legacy analysis only (1× Nyquist 125 kHz)'),
]

hdr = f'{"LoRa BW":>10}  {"decim_ratio":>11}  {"R":>5}  {"fs_out":>10}  {"Status"}'
print(hdr)
print('-' * len(hdr))
for bw, dr, R, fs, status in rows:
    print(f'{bw/1e3:>8.0f}kHz  {dr:>11d}  {R:>5d}  {fs:>8d}kS/s  {status}')

print()
print('Firmware startup: write decim_ratio = 0b01 (reg 0x12) for both BWs.')

---
## 5  Samples per Symbol

At R=128 (250 kS/s), the number of samples per LoRa symbol is:
- **250 kHz BW:** M = 2^SF exactly (1× Nyquist)
- **125 kHz BW:** M = 2^(SF+1) (2× oversampled — SX1302 compensates)

The integer-M property is essential for the SC detector (Schmidl-Cox requires exactly one symbol window = M samples).

In [ ]:
dec_r128 = SigmaDeltaDecimator(ratio=128)

print('R=128, fs_out=250 kS/s — samples per symbol M')
print(f'{"SF":>4}  {"250kHz BW":>10}  {"125kHz BW":>10}  {"2^SF":>8}  {"2^(SF+1)":>10}')
print('-' * 52)
for sf in range(6, 13):
    m_250 = dec_r128.samples_per_symbol(sf, bw_hz=250e3)
    m_125 = dec_r128.samples_per_symbol(sf, bw_hz=125e3)
    assert m_250 == 2**sf,     f'SF{sf} 250kHz: M={m_250} != 2^SF={2**sf}'
    assert m_125 == 2**(sf+1), f'SF{sf} 125kHz: M={m_125} != 2^(SF+1)={2**(sf+1)}'
    print(f'{sf:>4}  {m_250:>10d}  {m_125:>10d}  {2**sf:>8d}  {2**(sf+1):>10d}')

print()
print('PASS  M = 2^SF for 250 kHz BW, M = 2^(SF+1) for 125 kHz BW (all SF 6–12)')

---
## 6  Multi-Branch Coherence

The training accumulator and MRC combiner require all four receive branches to be sample-aligned. Any persistent offset between decimator instances would silently corrupt the cross-correlation.

**Required conditions (verified here for the Python model; must also hold in RTL):**
1. Identical inputs → identical outputs
2. All branches produce the same number of output samples (coincident `iq_valid`)

In [ ]:
rng = np.random.default_rng(42)
NR = 4
R  = 128
N_in = R * 256

# Shared ΣΔ bitstream (same signal, same phase — simulates same SX1257 front-end path)
tone_freq = 0.2 * FS_ADC / R / 2
t = np.arange(N_in) / FS_ADC
base_signal = 0.8 * np.sin(2 * np.pi * tone_freq * t)

decimators = [SigmaDeltaDecimator(ratio=R, output_bits=8, cic_only=True) for _ in range(NR)]
outputs = []
for j in range(NR):
    bits_j = sigma_delta_1st_order(base_signal) + 0j
    out_j  = decimators[j].process(bits_j)
    outputs.append(out_j)

# Check all outputs match branch 0
all_same    = all(np.allclose(outputs[j], outputs[0]) for j in range(1, NR))
same_length = all(len(outputs[j]) == len(outputs[0]) for j in range(1, NR))

print(f'Output length per branch: {[len(o) for o in outputs]}')
print(f'All branches identical:   {all_same}')
print(f'All branches same length: {same_length}')
assert all_same,    'Branches differ — coherence FAIL'
assert same_length, 'Branch lengths differ — coherence FAIL'
print('PASS  Multi-branch coherence: identical inputs → identical aligned outputs')

# Visual check: overlay a few output samples
fig, ax = plt.subplots(figsize=(10, 3))
n_show = 60
for j in range(NR):
    ax.plot(outputs[j][:n_show].real, lw=1.5, alpha=0.7, label=f'Branch {j}')
ax.set_xlabel('Sample index')
ax.set_ylabel('Output (normalised)')
ax.set_title('4-Branch Decimator Output (identical inputs) — curves should overlay exactly')
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7  DC Response and Saturation

A constant `+1` bitstream (maximum positive DC) should give output = +1.0 (full scale). The 8-bit output saturates at +127/128. The CIC takes a few output samples to settle; the initial transient is the integrator ramp.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

for ax, polarity, label in [
    (axes[0], +1.0, '+1 bitstream (max positive DC)'),
    (axes[1], -1.0, '−1 bitstream (max negative DC)'),
]:
    R = 128
    N_in = R * 64
    bits = np.full(N_in, polarity, dtype=np.complex128)
    dec  = SigmaDeltaDecimator(ratio=R, output_bits=8, cic_only=True)
    out  = dec.process(bits).real

    ax.plot(out, 'b.-', ms=3, lw=0.8)
    ax.axhline(polarity, color='r', ls='--', lw=1.5, label=f'Expected steady-state ({polarity:+.0f})')
    ax.axhline(polarity * 127/128, color='g', ls=':', lw=1.5, label=f'8-bit saturation (±127/128)')
    ax.set_xlabel('Output sample index')
    ax.set_ylabel('Output value (normalised)')
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Verify steady state
    steady = out[10:]  # skip initial transient
    assert np.allclose(steady, polarity * 127/128, atol=0.01), \
        f'DC steady-state wrong: mean={np.mean(steady):.4f}, expected {polarity * 127/128:.4f}'
    print(f'PASS  DC {polarity:+.0f}: steady-state = {np.mean(steady):.4f}  (expected {polarity*127/128:.4f})')

plt.tight_layout()
plt.show()

---
## 8  LoRa Chirp Loopback

Feed a LoRa upchirp symbol through a 1st-order ΣΔ ADC and CIC decimator. The output should reproduce the chirp with droop from the CIC passband roll-off (no FIR correction in the deployed path).

In [ ]:
SF   = 7
BW   = 250e3   # 250 kHz LoRa
R    = 128     # deployed setting
M    = 2 ** SF # 128 samples per symbol at fs_out

# Generate one upchirp at decimated rate, then upsample R× for the ΣΔ input
chirp_bb = modulate(0, M)                          # complex upchirp at 250 kS/s
chirp_hi = np.repeat(chirp_bb, R) / np.abs(chirp_bb.max())  # upsample to 32 MS/s, normalise

# ΣΔ modulate: I and Q independently
amp = 0.8
bits_i = sigma_delta_1st_order(amp * chirp_hi.real)
bits_q = sigma_delta_1st_order(amp * chirp_hi.imag)
bits   = bits_i + 1j * bits_q

# Decimate
dec_deployed = SigmaDeltaDecimator(ratio=R, output_bits=8, cic_only=True)
dec_fir      = SigmaDeltaDecimator(ratio=R, output_bits=8, cic_only=False)
out_cic      = dec_deployed.process(bits)
out_fir      = dec_fir.process(bits)

# Align to chirp by removing CIC group delay (≈ R*N/2 input samples → N/2 output samples)
gd = 3 * (R - 1) // (2 * R) + 1   # rough group delay in output samples

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
t_bb = np.arange(M) / (FS_ADC / R) * 1e3   # ms
t_out = np.arange(len(out_cic)) / (FS_ADC / R) * 1e3

# IQ time domain
for row, component, ref_fn in [
    (0, 'I', lambda x: x.real),
    (1, 'Q', lambda x: x.imag),
]:
    ax = axes[row, 0]
    ax.plot(t_bb, ref_fn(chirp_bb) * amp, 'k--', lw=1.5, label='Reference (ideal)', alpha=0.7)
    ax.plot(t_out[:M], ref_fn(out_cic)[:M], 'b-', lw=1.5, label='CIC-only (deployed)')
    ax.plot(t_out[:M], ref_fn(out_fir)[:M], 'r--', lw=1.5, label='CIC + FIR')
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel(f'{component} (normalised)')
    ax.set_title(f'Chirp {component} channel')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Instantaneous frequency (angle derivative)
ax = axes[0, 1]
ref_phase  = np.unwrap(np.angle(chirp_bb))
cic_phase  = np.unwrap(np.angle(out_cic[:M]))
fir_phase  = np.unwrap(np.angle(out_fir[:M]))
ax.plot(t_bb[1:], np.diff(ref_phase)  / (2*np.pi) * (FS_ADC/R) / 1e3, 'k--', label='Reference', alpha=0.7)
ax.plot(t_bb[1:], np.diff(cic_phase)  / (2*np.pi) * (FS_ADC/R) / 1e3, 'b-',  label='CIC-only')
ax.plot(t_bb[1:], np.diff(fir_phase)  / (2*np.pi) * (FS_ADC/R) / 1e3, 'r--', label='CIC+FIR')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Instantaneous frequency (kHz)')
ax.set_title('Chirp instantaneous frequency')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Constellation (I vs Q)
ax = axes[1, 1]
n_plot = M
c = plt.cm.viridis(np.linspace(0, 1, n_plot))
ax.scatter(out_cic[:n_plot].real, out_cic[:n_plot].imag, c=c, s=4, label='CIC-only')
ax.set_xlabel('I'); ax.set_ylabel('Q')
ax.set_title('CIC output IQ (colour = time, blue→yellow)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

fig.suptitle(f'LoRa SF{SF} Upchirp — ΣΔ → CIC Loopback (R={R})', fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/sd_decimator_loopback.png', bbox_inches='tight')
plt.show()
print('Saved: sim/plots/sd_decimator_loopback.png')

---
## 9  FIR Upgrade Gate Check

From `planning/blocks/ΣΔ Decimator.md`, the optional TDM+FIR upgrade should only be taken if post-PNR utilisation is ≤55% and ≥86 kµm² headroom remains. This cell quantifies the SQNR improvement the FIR buys.

In [ ]:
print('SQNR comparison at 0.4×Nyquist, R=128 (Python model):')
print(f'{"":<6}  {"CIC-only":>10}  {"CIC+FIR":>10}  {"Gain":>8}')
print('-' * 40)

for R in [128, 256]:
    dec_cic = SigmaDeltaDecimator(ratio=R, output_bits=16, cic_only=True)
    dec_fir = SigmaDeltaDecimator(ratio=R, output_bits=16, cic_only=False)

    fs_out = FS_ADC / R
    f_tone = 0.4 * fs_out / 2
    n_out  = 512 * int(round(fs_out / f_tone))
    n_in   = n_out * R
    t_in   = np.arange(n_in) / FS_ADC
    bits   = sigma_delta_1st_order(0.9 * np.sin(2 * np.pi * f_tone * t_in)) + 0j

    def sqnr_from_output(dec):
        out  = dec.process(bits).real
        skip = 20
        y    = out[skip:]
        omega = 2 * np.pi * f_tone / fs_out
        n    = np.arange(len(y))
        C    = np.column_stack([np.cos(omega*n), np.sin(omega*n)])
        cf, _, _, _ = np.linalg.lstsq(C, y, rcond=None)
        sig_amp = np.sqrt(cf[0]**2 + cf[1]**2)
        nse_rms = float(np.std(y - C @ cf))
        return 20*np.log10(sig_amp/nse_rms) if nse_rms > 0 else 99.0

    sq_cic = sqnr_from_output(dec_cic)
    sq_fir = sqnr_from_output(dec_fir)
    print(f'R={R:<4}  {sq_cic:>8.1f} dB  {sq_fir:>8.1f} dB  {sq_fir-sq_cic:>+6.1f} dB')

print()
print('RTL benchmark (job 1102):')
print('  R=128 CIC-only: 30.6 dB, CIC+FIR: 37.7 dB  → +7.1 dB gain from FIR')
print('  FIR upgrade justified only if area/timing headroom exists after top-level P&R.')

---
## Summary

| Item | Value |
|---|---|
| Deployed config | CIC-only, N=3, R=128 (`decim_ratio=1`) |
| fs_out | 250 kS/s |
| Supported LoRa BWs | 125 kHz (2× OS) and 250 kHz (1× Nyquist) |
| Deployed SQNR (RTL) | 30.6 dB ≥ 28 dB threshold ✓ |
| 500 kHz BW | NOT supported — CIC SQNR 9.6 dB at R=64 |
| FIR upgrade gain | +7 dB SQNR; take only if P&R headroom allows |
| RTL authoritative | Alias noise at R=64 not captured by Python model |